In [ ]:
import math
import pandas as pd
import numpy as np
import timeit


def H(df: pd.DataFrame, Y: str, X: str = "") -> np.float64:
    if X == "":
        return -(df[Y].value_counts(normalize=True)).apply(lambda x: x * math.log2(x)).sum()
    else:
        cond_df = df[X].value_counts(normalize=True).reset_index()
        return (cond_df["proportion"] * cond_df[X].apply(lambda x: H(df[df[X] == x], Y))).sum()
    
def g(df: pd.DataFrame, Y: str, X: str) -> np.float64:
    return H(df, Y) - H(df, Y, X)

def g_R(df: pd.DataFrame, Y: str, X: str) -> np.float64:
    return g(df, Y, X) / H(df, X)

def Gini(df: pd.DataFrame, Y: str) -> np.float64:
    return df[Y].value_counts(normalize=True).apply(lambda x: x * (1 - x)).sum()




df = pd.DataFrame({
    "年龄": ["青年", "青年", "青年", "青年", "青年", "中年", "中年", "中年", "中年", "中年", "老年", "老年", "老年", "老年", "老年"],
    "有工作": ["否", "否", "是", "是", "否", "否", "否", "是", "否", "否", "否", "否", "是", "是", "否"],
    "有自己的房子": ["否", "否", "否", "是", "否", "否", "否", "是", "是", "是", "是", "是", "否", "否", "否"],
    "信贷情况": ["一般", "好", "好", "一般", "一般", "一般", "好", "好", "非常好", "非常好", "非常好", "好", "好", "非常好", "一般"],
    "类别": ["否", "否", "是", "是", "否", "否", "否", "是", "是", "是", "是", "是", "是", "是", "否"]
})

# stmt = lambda: g(df, "类别", "年龄")
# timer = timeit.Timer(stmt)
# execution_time = timer.timeit(number=1000)
# print(execution_time)

# print(g(df, "类别", "年龄"))
# 0.08300749985576883

# print(g(df, "类别", "有工作"))
# 0.32365019815155627

# print(g(df, "类别", "有自己的房子"))
# 0.4199730940219749

# print(g(df, "类别", "信贷情况"))
# np.float64(0.36298956253708536)

Gini(df, "类别")

np.float64(0.48)

In [38]:
df["年龄"].value_counts(normalize=True).reset_index()

,年龄,proportion
0,青年,0.333333
1,中年,0.333333
2,老年,0.333333


In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, value=None, left=None, right=None, children=None):
        self.feature = feature
        
        # ID3、C4.5算法专用
        self.children = children

        # CART算法专用
        self.threshold = threshold
        self.left = left
        self.right = right

        # 叶子节点专用
        self.value = value
        


class DecisionTree:
    def __init__(self, criterion='id3', max_depth=None, min_samples_split=2):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None
        self.classes = None

    def fit(self, df, target):
        self.classes = df[target].unique()
        features = df.columns.drop(target).tolist()
        self.tree = self._build_tree(df, target, features, depth=0)

    def _build_tree(self, df, target, features, depth):
        if len(df[target].unique()) == 1:
            return Node(value=df[target].iloc[0])
        if len(features) == 0 or len(df) < self.min_samples_split:
            return Node(value=df[target].mode()[0])
        if self.max_depth is not None and depth >= self.max_depth:
            return Node(value=df[target].mode()[0])

        if self.criterion == 'id3':
            best_feature = self._best_split_id3(df, target, features)
            if best_feature is None:
                return Node(value=df[target].mode()[0])
            node = Node(feature=best_feature)
            children = {}
            for val in df[best_feature].unique():
                subset = df[df[best_feature] == val]
                if len(subset) == 0:
                    children[val] = Node(value=df[target].mode()[0])
                else:
                    remaining_features = [f for f in features if f != best_feature]
                    children[val] = self._build_tree(subset, target, remaining_features, depth + 1)
            node.children = children
            return node

        elif self.criterion == 'c4.5':
            best_feature = self._best_split_c45(df, target, features)
            if best_feature is None:
                return Node(value=df[target].mode()[0])
            node = Node(feature=best_feature)
            children = {}
            for val in df[best_feature].unique():
                subset = df[df[best_feature] == val]
                if len(subset) == 0:
                    children[val] = Node(value=df[target].mode()[0])
                else:
                    remaining_features = [f for f in features if f != best_feature]
                    children[val] = self._build_tree(subset, target, remaining_features, depth + 1)
            node.children = children
            return node

        elif self.criterion == 'cart':
            best_feature, best_value = self._best_split_cart(df, target, features)
            if best_feature is None:
                return Node(value=df[target].mode()[0])
            node = Node(feature=best_feature, threshold=best_value)
            left_df = df[df[best_feature] == best_value]
            right_df = df[df[best_feature] != best_value]
            if len(left_df) == 0:
                node.left = Node(value=df[target].mode()[0])
            else:
                node.left = self._build_tree(left_df, target, features, depth + 1)
            if len(right_df) == 0:
                node.right = Node(value=df[target].mode()[0])
            else:
                node.right = self._build_tree(right_df, target, features, depth + 1)
            return node

        else:
            raise ValueError("criterion must be 'id3', 'c4.5', or 'cart'")

    def _best_split_id3(self, df, target, features):
        gains = [(f, g(df, target, f)) for f in features]
        max_gain = max(gains, key=lambda x: x[1])[1]
        if max_gain <= 0:
            return None
        return max(gains, key=lambda x: x[1])[0]

    def _best_split_c45(self, df, target, features):
        gains_ratio = [(f, g_R(df, target, f)) for f in features]
        max_ratio = max(gains_ratio, key=lambda x: x[1])[1]
        if max_ratio <= 0:
            return None
        return max(gains_ratio, key=lambda x: x[1])[0]

    def _best_split_cart(self, df, target, features):
        best_gini = float('inf')
        best_feature = None
        best_value = None
        for f in features:
            for val in df[f].unique():
                left_df = df[df[f] == val]
                right_df = df[df[f] != val]
                if len(left_df) == 0 or len(right_df) == 0:
                    continue
                gini_left = Gini(left_df, target)
                gini_right = Gini(right_df, target)
                weighted_gini = (len(left_df) * gini_left + len(right_df) * gini_right) / len(df)
                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = f
                    best_value = val
        return best_feature, best_value

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            return X.apply(lambda row: self._predict_one(row), axis=1).values
        else:
            raise TypeError("X should be a pandas DataFrame")

    def _predict_one(self, row):
        node = self.tree
        while node.value is None:
            if self.criterion in ['id3', 'c4.5']:
                val = row[node.feature]
                if val in node.children:
                    node = node.children[val]
                else:
                    return list(node.children.values())[0].value
            elif self.criterion == 'cart':
                if row[node.feature] == node.threshold:
                    node = node.left
                else:
                    node = node.right
        return node.value